# 1. Install deps

In [4]:
!pip install pandas

# 2. Load data та  3. Implement/load rules + dictionaries

In [5]:
import sys
import os

if 'google.colab' in sys.modules:
    if not os.path.exists('/content/NLP'):
        !git clone -b lab-4 https://github.com/AndrianaNahirna/NLP.git

    %cd /content/NLP

    !pip install -r requirements.txt -q
    !git fetch origin
    !git reset --hard origin/lab-4

    sys.path.append('/content/NLP')

print("Середовище налаштовано.")

/content/NLP
HEAD is now at a144be4 Add 30 edge cases
Середовище налаштовано.


# 4. Run extraction on sample

In [6]:
import json
import os
import pandas as pd
from IPython.display import display
from sentiment.src.ie_rules import RuleBasedExtractor

print("Тестування Edge Cases\n")

base_dir = 'sentiment' if os.path.exists('sentiment') else '.'
edge_cases_path = os.path.join(base_dir, 'tests/ie_edge_cases.jsonl')
extractor = RuleBasedExtractor(resources_path='sentiment/resources')

results_data = []

with open(edge_cases_path, 'r', encoding='utf-8') as f:
    for line in f:
        case = json.loads(line)
        text = case['raw_text']
        expected = case['expected_behavior']
        extracted = extractor.extract_all(text)

        if extracted:
            extracted_str = ", ".join([f"{e['value']} [{e['field_type']}]" for e in extracted])
        else:
            extracted_str = "Нічого"

        results_data.append({
            "Текст (raw_text)": text,
            "Знайдені сутності": extracted_str,
            "Очікувана поведінка": expected
        })

df_edges = pd.DataFrame(results_data)

pd.set_option('display.max_colwidth', None)
display(df_edges.head(15))

Тестування Edge Cases



,Текст (raw_text),Знайдені сутності,Очікувана поведінка
0,"Оновив додаток до версії 2.0.12, працює краще",Нічого,Не вважати номер версії ПЗ датою
1,Модель телефону 10.12 дуже стара,Нічого,Неповна дата (без року) не повинна витягуватись
2,Спробував ввести 32.01.2023,Нічого,Ігнорувати неіснуючі дні (більше 31)
3,Помилка в базі: 15.13.2020,Нічого,Ігнорувати неіснуючі місяці (більше 12)
4,Дата покупки 00.05.2019,Нічого,"Нульовий день не є валідним, ігнорувати"
5,Отримав 12.04.16 на пошті,2016-04-12 [DATE],Правильно додати століття до двозначного року (2016)
6,Замовляв 5 Січня 2024,2024-01-05 [DATE],Розпізнати текстовий місяць незалежно від регістру
7,Приїде 1.2.2023,2023-02-01 [DATE],Розпізнати дату без нулів на початку дня та місяця
8,Шум 12 . 04 . 2020,Нічого,"Не вважати датою, якщо є зайві пробіли навколо крапок"
9,Купив 31.02.2022,2022-02-31 [DATE],"В ідеалі ігнорувати, але наш базовий regex пропустить (допустимий edge case для покращення)"


# 5. Evaluate on gold subset

In [8]:
import pandas as pd
import json
from sentiment.src.ie_rules import RuleBasedExtractor

gold_records = []
with open('sentiment/data/sample/lab4_gold_ie.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        gold_records.append(json.loads(line))

gold_df = pd.DataFrame(gold_records)

extractor = RuleBasedExtractor(resources_path='sentiment/resources')

results = {"DATE": {"TP": 0, "FP": 0}, "AMOUNT": {"TP": 0, "FP": 0}, "LOCATION": {"TP": 0, "FP": 0}}

grouped = gold_df.groupby('text_id')

for text_id, group in grouped:
    text = group['text'].iloc[0]
    extracted = extractor.extract_all(text)

    gold_values = {row['field_type']: set() for _, row in group.iterrows()}
    for _, row in group.iterrows():
        gold_values[row['field_type']].add(row['normalized_value'])

    for ext in extracted:
        ftype = ext['field_type']
        val = ext['value']

        if ftype in gold_values and val in gold_values[ftype]:
            results[ftype]["TP"] += 1
        else:
            results[ftype]["FP"] += 1

print("\n    ТАБЛИЦЯ PRECISION")
for ftype, counts in results.items():
    tp = counts["TP"]
    fp = counts["FP"]
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    print(f"{ftype:10} | Precision: {precision:.2f} ({tp} правильних / {tp+fp} знайдених)")

print("\nКоментар: Оскільки ми використовуємо підхід 'precision-first' з жорсткими обмеженнями (дні від 1 до 31, фіксований словник міст), на еталонній вибірці ми отримуємо максимальну точність 1.0 (100%).")


    ТАБЛИЦЯ PRECISION
DATE       | Precision: 1.00 (17 правильних / 17 знайдених)
AMOUNT     | Precision: 1.00 (17 правильних / 17 знайдених)
LOCATION   | Precision: 1.00 (20 правильних / 20 знайдених)

Коментар: Оскільки ми використовуємо підхід 'precision-first' з жорсткими обмеженнями (дні від 1 до 31, фіксований словник міст), на еталонній вибірці ми отримуємо максимальну точність 1.0 (100%).


# 6. Error analysis

## Аналіз помилок

Під час розробки регулярних виразів та словників на всьому корпусі відгуків було виявлено низку хибних спрацювань (False Positives). Нижче наведено 10 типових помилок та відредагованих правил, які застосовано для їх усунення.

### Поле DATE (Дати)
**1. Версії прошивок/ПЗ**
* **Що спрацювало:** Текст "Оновив до версії 2.0.12." розпізнався як дата `2012-00-02`.
* **Чому це помилка:** Це номер версії програмного забезпечення, а не дата.
* **Яке правило/анти-правило додали:** Додано логічну перевірку в Python: `if 1 <= int(m) <= 12 and 1 <= int(d) <= 31:`.

**2. Неповні дати (номери моделей)**
* **Що спрацювало:** Текст "Купив модель 10.12" розпізнався як дата `2012-10-00`.
* **Чому це помилка:** Відсутній рік, це просто номер моделі з крапкою.
* **Яке правило/анти-правило додали:** У regex додано жорстку вимогу наявності року (2 або 4 цифри) після другої крапки: `\.(20\d{2}|19\d{2}|\d{2})\b`.

**3. Неіснуючі дні в календарі**
* **Що спрацювало:** Опечатка "Дата 32.01.2023" дала `2023-01-32`.
* **Чому це помилка:** У місяці максимум 31 день.
* **Яке правило/анти-правило додали:** Обмеження `1 <= int(d) <= 31` відфільтрувало цей кейс.

**4. Нульові значення днів/місяців**
* **Що спрацювало:** Текст "00.05.2019" розпізнався як `2019-05-00`.
* **Чому це помилка:** Нульового дня не існує.
* **Яке правило/анти-правило додали:** Додано умову строго більше нуля: `1 <= int(d)`.

### Поле AMOUNT (Суми)
**5. Відсотки замість сум**
* **Що спрацювало:** Текст "Ціна впала на 100%." дав `100.0 UNKNOWN`.
* **Чому це помилка:** Знак відсотка не є валютою.
* **Яке правило/анти-правило додали:** Створено жорсткий словник `currencies.json`. Будь-які символи, яких там немає (наприклад, `%`), ігноруються.

**6. Кількість товару**
* **Що спрацювало:** Текст "Взяв 2 шт за 50" розпізнався як `2.0 UNKNOWN` та `50.0 UNKNOWN`.
* **Чому це помилка:** Це кількість штук та число без вказівки валюти.
* **Яке правило/анти-правило додали:** Regex тепер вимагає, щоб одразу після числа обов'язково йшов один із ключів словника валют: `\s*({curr_keys})\b`.

**7. Об'єм пам'яті / Технічні характеристики**
* **Що спрацювало:** Текст "Пам'ять 64 Гб" дав `64.0 UNKNOWN`.
* **Чому це помилка:** "Гб" — це одиниця виміру інформації, а не грошей.
* **Яке правило/анти-правило додали:** Оскільки "Гб" відсутнє в нашому `currencies.json`, правило ігнорує цей збіг.

**8. Дробові суми з комою**
* **Що спрацювало:** Текст "заплатив 1,5 грн" не знаходився взагалі (False Negative) або ламався парсер.
* **Чому це помилка:** В україномовному середовищі кома часто використовується замість крапки.
* **Яке правило/анти-правило додали:** Regex розширено для підтримки обох розділювачів `[.,]\d{1,2}`, а перед кастом у float додано `.replace(',', '.')`.

### Поле LOCATION (Міста)
**9. Прикметники від назв міст**
* **Що спрацювало:** Текст "Київський торт" витягнув `Київ`.
* **Чому це помилка:** Це прикметник, який є частиною назви продукту.
* **Яке правило/анти-правило додали:** У regex для міст додано маркери меж слова `\b`, тому слова з іншими закінченнями або суфіксами не проходять фільтр.

**10. Назви областей**
* **Що спрацювало:** Текст "Львівська область" витягнув `Львів`.
* **Чому це помилка:** Область — це ширший регіон, а не саме місто.
* **Яке правило/анти-правило додали:** Використовується строгий пошук за нормалізованими формами та відмінками зі словника `locations.json` із межами слова `\b`. "Львівська" не збігається з жодним відмінком слова "Львів" у нашому словнику.

# 7. Save/update docs/audit_summary_lab4.md

In [9]:
import os
import json

os.makedirs('tests', exist_ok=True)
os.makedirs('docs', exist_ok=True)
os.makedirs('labs/lab04', exist_ok=True)

audit_md = """# Audit Summary: Lab 4 (Rule-based IE)

## 1. Загальна інформація
* **Напрям:** Rule-based Information Extraction (витяг сутностей за правилами).
* **Сутності:** `DATE` (Дати), `AMOUNT` (Суми), `LOCATION` (Міста).
* **Підхід:** Precision-first (висока точність за рахунок regex та словників).

## 2. Метрики Precision на Gold Subset
* **DATE:** 1.0 (100%)
* **AMOUNT:** 1.0 (100%)
* **LOCATION:** 1.0 (100%)

## 3. Аналіз помилок (False Positives) - 10 кейсів
Під час розробки на всьому корпусі було виявлено такі типові хибні спрацювання (False Positives), для яких ми **додали правила-запобіжники**:

1. **Текст:** "Оновив до версії 2.0.12." -> **FP:** `2012-00-02` (DATE). **Рішення:** Додано перевірку, що місяць <= 12, а день <= 31.
2. **Текст:** "Ціна впала на 100%." -> **FP:** `100.0 UNKNOWN` (AMOUNT). **Рішення:** Введено жорсткий словник валют, відсотки ігноруються.
3. **Текст:** "Модель 10.12" -> **FP:** `2012-10-00` (DATE). **Рішення:** Regex вимагає повного року (2 або 4 цифри) після другої крапки.
4. **Текст:** "Взяв 2 шт за 50" -> **FP:** `2.0 UNKNOWN` (AMOUNT). **Рішення:** Числа без ідентифікатора валюти поруч ігноруються.
5. **Текст:** "Дата 32.01.2023" -> **FP:** `2023-01-32` (DATE). **Рішення:** Додано `if 1 <= int(d) <= 31`.
6. **Текст:** "Київський торт" -> **FP:** `Київ` (LOCATION). **Рішення:** Пошук здійснюється за точним збігом слів (word boundaries `\\b`), прикметники ігноруються.
7. **Текст:** "00.05.2019" -> **FP:** `2019-05-00` (DATE). **Рішення:** Умова `1 <= int(d)`.
8. **Текст:** "Пам'ять 64 Гб" -> **FP:** `64.0 UNKNOWN` (AMOUNT). **Рішення:** "Гб" не входить до `currencies.json`.
9. **Текст:** "Львівська область" -> **FP:** `Львів` (LOCATION). **Рішення:** Витягується тільки точна назва міста зі словника.
10. **Текст:** "1.5.грн" -> **FP:** `5.0 UAH` (AMOUNT). **Рішення:** Regex вдосконалено для підтримки плаваючої коми `[.,]\\d{1,2}`.
"""
with open('docs/audit_summary_lab4.md', 'w', encoding='utf-8') as f: f.write(audit_md)




print("Файл audit_summary_lab4.md згенерований.")

Файл audit_summary_lab4.md згенерований.
